# Clustering and Unsupervised Learning

<a target="_blank" href="https://colab.research.google.com/github/imamitjain/notebooks/blob/main/02-ml-fundamentals/03_clustering_and_unsupervised.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objective:** Apply K-Means, DBSCAN, and hierarchical clustering. Evaluate clusters with silhouette scores and visualize results with PCA/t-SNE.

**Prerequisites:** Classification basics (notebook 02)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q numpy pandas matplotlib scikit-learn scipy


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs

## 1. Generate Clustered Data

In [ ]:
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', alpha=0.6, edgecolors='k', linewidth=0.5)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Ground Truth Clusters')
plt.colorbar(label='Cluster')
plt.show()

## 2. K-Means Clustering

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
y_kmeans = kmeans.fit_predict(X)

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=y_kmeans, cmap='viridis', alpha=0.6, edgecolors='k', linewidth=0.5)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c='red', marker='X', s=200, edgecolors='black', linewidth=2, label='Centroids')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title(f'K-Means (k=4) — Silhouette: {silhouette_score(X, y_kmeans):.3f}')
plt.legend()
plt.show()

## 3. The Elbow Method and Silhouette Score

In [ ]:
inertias = []
silhouettes = []
K = range(2, 10)

for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(K, inertias, 'bo-')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(K, silhouettes, 'ro-')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')

plt.tight_layout()
plt.show()
print(f"Best k by silhouette: {list(K)[np.argmax(silhouettes)]}")

## 4. DBSCAN (Density-Based Clustering)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

dbscan = DBSCAN(eps=0.5, min_samples=5)
y_dbscan = dbscan.fit_predict(X_scaled)

n_clusters = len(set(y_dbscan)) - (1 if -1 in y_dbscan else 0)
n_noise = list(y_dbscan).count(-1)

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=y_dbscan, cmap='viridis', alpha=0.6, edgecolors='k', linewidth=0.5)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title(f'DBSCAN — {n_clusters} clusters, {n_noise} noise points')
plt.colorbar(label='Cluster')
plt.show()

## 5. Hierarchical Clustering

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

agg = AgglomerativeClustering(n_clusters=4)
y_agg = agg.fit_predict(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X[:, 0], X[:, 1], c=y_agg, cmap='viridis', alpha=0.6, edgecolors='k', linewidth=0.5)
axes[0].set_title(f'Agglomerative (k=4) — Silhouette: {silhouette_score(X, y_agg):.3f}')

Z = linkage(X[:50], method='ward')
dendrogram(Z, ax=axes[1], leaf_rotation=90)
axes[1].set_title('Dendrogram (first 50 samples)')

plt.tight_layout()
plt.show()

## 6. Dimensionality Reduction for Visualization (PCA and t-SNE)

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X_digits, y_digits = digits.data, digits.target

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_digits)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_digits)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_digits, cmap='tab10', alpha=0.5, s=10)
axes[0].set_title(f'PCA (explained var: {pca.explained_variance_ratio_.sum():.1%})')
plt.colorbar(scatter1, ax=axes[0])

scatter2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_digits, cmap='tab10', alpha=0.5, s=10)
axes[1].set_title('t-SNE')
plt.colorbar(scatter2, ax=axes[1])

plt.tight_layout()
plt.show()

## Try It Yourself

1. Generate blobs with varying densities. Compare K-Means vs DBSCAN — which handles non-uniform density better?
2. Apply PCA to a high-dimensional dataset, plot the explained variance ratio, and determine how many components capture 95% of variance.
3. Use t-SNE to visualize the digits dataset (`sklearn.datasets.load_digits`) colored by label.

In [ ]:
# Your code here